# G1 Academy 3 - Solution: direct Dex3 and arm control


## Introduction
ChannelPublisher writes typed HandCmd messages to a Dex3 command topic; ChannelSubscriber reads HandState tactile data. G1ArmActionClient triggers named high-level actions. util provides only hand target arrays and joint-group mapping, not a hidden hand client.

## Direct hand-message model

A Dex3 HandCmd message contains one motor command per hand joint. For every commanded joint, set the control mode and desired position plus conservative dq, tau, kp, and kd values, then write the complete typed message through the single approved publisher. HandState arrives independently and contains motor state plus press_sensor_state arrays. A tactile threshold is empirical: measure no-object noise first, then calibrate on a supervised safe grasp.

High-level arm actions use G1ArmActionClient. They should never run concurrently with a low-level arm command stream unless an explicit controller handoff has been completed.


## Task 1 - Create direct tactile subscriber and hand publisher
HandState_ exposes motor and press_sensor_state fields. HandCmd_ defines the topic type; the native HandCmd message is filled for every finger joint. Keep ownership of one hand command publisher.


In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__HandCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import HandCmd_, HandState_
from util import HAND_JOINT_NAMES, HAND_OPEN, HAND_CLOSED
hand_state=None
def on_hand_state(msg):
    global hand_state; hand_state=msg
right_pub=ChannelPublisher("rt/dex3/right/cmd", HandCmd_); right_pub.Init()
right_sub=ChannelSubscriber("rt/dex3/right/state", HandState_); right_sub.Init(on_hand_state, 20)
def tactile_samples():
    return [float(v) for sensor in getattr(hand_state, "press_sensor_state", []) for v in sensor.pressure] if hand_state else []
# print(tactile_samples())


## Task 2 - Implement gradual tactile-aware close
Use HAND_OPEN/CLOSED target maps from util, but construct/write every native command frame yourself. Establish a no-object baseline and calibrated threshold; stop early when tactile pressure reaches it.


In [ ]:
def write_hand(targets, publisher=right_pub, kp=0.8, kd=0.05):
    msg=unitree_hg_msg_dds__HandCmd_()
    for i, q in enumerate(targets):
        cmd=msg.motor_cmd[i]; cmd.mode=(i & 15) | 16; cmd.q=float(q); cmd.dq=0.0; cmd.tau=0.02; cmd.kp=kp; cmd.kd=kd
    publisher.Write(msg)
def gradual_close(threshold, steps=40, delay_s=0.05):
    for step in range(1, steps+1):
        a=step/steps; frame=[x+(y-x)*a for x,y in zip(HAND_OPEN["right"], HAND_CLOSED["right"])]
        write_hand(frame); time.sleep(delay_s)
        if tactile_samples() and max(tactile_samples()) >= threshold: return {"contact":True,"step":step}
    return {"contact":False,"step":steps}


## Task 3 - Use direct high-level arm client
G1ArmActionClient invokes documented action IDs. util may provide a mapping during the lesson; do not combine high-level and low-level command ownership without an explicit handoff.

Task 3 now covers all requested arm concepts. High-level gestures use G1ArmActionClient action IDs for clap, face wave, and release. Teach captures the current direct lowstate arm-joint pose; repeat interpolates that saved pose through the direct rt/arm_sdk publisher; extend replays a separately saved forward/extended pose slowly. This is joint-pose interpolation, not a replacement for the supplied end-effector IK pipeline. Keep one arm command owner at a time.


In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.g1.arm.g1_arm_action_client import G1ArmActionClient
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_, LowState_
from unitree_sdk2py.utils.crc import CRC
from util import JOINT_GROUPS

_latest_lowstate = None
def on_lowstate(message):
    global _latest_lowstate
    _latest_lowstate = message
lowstate_sub = ChannelSubscriber("rt/lowstate", LowState_); lowstate_sub.Init(on_lowstate, 10)
arm_pub = ChannelPublisher("rt/arm_sdk", LowCmd_); arm_pub.Init()
_arm_crc = CRC()
def teach_pose(side="right"):
    if _latest_lowstate is None: raise RuntimeError("Wait for fresh rt/lowstate.")
    return {joint: float(_latest_lowstate.motor_state[joint].q) for joint in JOINT_GROUPS[side + "_arm"]}
def write_arm_pose(targets, weight=1.0):
    message = unitree_hg_msg_dds__LowCmd_(); message.mode_pr = 0; message.mode_machine = 0
    message.motor_cmd[29].q = float(weight)
    for joint, q in targets.items():
        cmd=message.motor_cmd[joint]; cmd.mode=1; cmd.q=float(q); cmd.dq=0.0; cmd.tau=0.0; cmd.kp=30.0; cmd.kd=1.5
    message.crc = _arm_crc.Crc(message); arm_pub.Write(message)
def repeat_pose(saved_pose, duration_s=4.0, steps=100):
    start=teach_pose("right")
    for step in range(1, steps+1):
        alpha=step/steps
        write_arm_pose({j:start[j]+(saved_pose[j]-start[j])*alpha for j in saved_pose})
        time.sleep(duration_s/steps)
def extend_arm_forward(extended_pose):
    return repeat_pose(extended_pose, duration_s=8.0, steps=200)
arm=G1ArmActionClient(); arm.SetTimeout(10.0); arm.Init()
def gesture(name):
    return arm.ExecuteAction({"clap":17, "face_wave":25, "release":99}[name])


### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.
